# Study 825 — Oil Predicts Equities 🛢️📉

**Does *this month's* move in the oil price forecast *next month's* stock market — negatively?**

Driesprong, Jacobsen & Maat (2008), *"Striking Oil: Another Puzzle?"*, report that a rise
in the oil price this month is followed by **lower** equity returns next month: oil is a
slow-diffusing macro shock that the stock market prices in with a lag. We take the
self-contained monthly version — a predictive regression of the S&P 500 (**SPY**)
*forward* one-month return on the *trailing* one-month oil (**USO**) return
(2006-05-31 → 2026-05-31, 241 months) — and stamp it on the real tape.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint
`effe48bc1a4f`); the live cells run the fast synthetic control. Survivorship: USO/SPY
are continuously-listed ETFs — no delisting bias on this pair, named on the Signal axis.*


## 1. The idea in one picture

Oil is an input cost to almost every listed company and a barometer of global demand. Driesprong et al argue investors are *slow* to price an oil shock, so a rising oil price this month quietly drags on stocks **next** month. The test is a one-line forecast regression: today's oil return on the left of a one-month gap, tomorrow's equity return on the right. The claim fixes the sign: the slope should be **negative**.

In [1]:
import numpy as np
R = dict(beta=0.0136, t_nw=0.33, r2_pct=0.11, fwd_down_pct=0.95, fwd_up_pct=0.83)
print('predictive slope beta = %+.4f  (Newey-West t = %+.2f,  R2 = %.2f%%)'
      % (R['beta'], R['t_nw'], R['r2_pct']))
print('  Driesprong predicts beta < 0; on the real tape it is ~0 and slightly POSITIVE')
print('  next-month SPY after oil FELL: %+.2f%%   after oil ROSE: %+.2f%%'
      % (R['fwd_down_pct'], R['fwd_up_pct']))

predictive slope beta = +0.0136  (Newey-West t = +0.33,  R2 = 0.11%)
  Driesprong predicts beta < 0; on the real tape it is ~0 and slightly POSITIVE
  next-month SPY after oil FELL: +0.95%   after oil ROSE: +0.83%


## 2. Is the machinery honest? A live synthetic control

We plant the Driesprong link in a seeded toy tape (`edge>0` → a *negative* oil→equity slope) and check the regression recovers it with the right sign — and stays *silent* on the null (`edge=0`, oil and equities independent). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from oil_equities import data, strategy as st
null = st.synthetic_detect(data.synthetic_series(edge=0.0, seed=825))
planted = st.synthetic_detect(data.synthetic_series(edge=0.35, seed=825))
print('null world   : slope t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: slope t = %+.2f  beta = %+.4f  (should light up NEGATIVE)'
      % (planted['t_nw'], planted['beta']))

null world   : slope t = -0.57  (should be ~0)
planted world: slope t = -5.91  beta = -0.1951  (should light up NEGATIVE)


## 3. The honest verdict — the famous edge does *not* replicate

On 2006–2026 US ETFs the predictive slope is **+0.0136** with a Newey-West *t* of just **+0.33** and an R² of **0.11%** — statistically indistinguishable from zero, and if anything the *wrong* (positive) sign versus the claimed negative one. Next-month SPY after the oil-up months (+0.83%) is barely different from after oil-down months (+0.95%). A 2,000-draw permutation placebo puts the observed slope at p = 0.59 (pure noise). The seeded synthetic control recovers a *planted* negative slope cleanly, so this is a real null, not a broken engine. **Signal: None.** And no version of the timer beats simply holding SPY, so **Tradability: Mirage.**